In [3]:
# %%catch — wraps a cell's execution so an exception can never halt "Run All".
# Uses IPython's run_cell (not a plain exec) so the normal traceback and the
# cell's last-expression auto-display still behave exactly like a normal cell.
from IPython.core.magic import register_cell_magic


@register_cell_magic
def catch(line, cell):
    result = get_ipython().run_cell(cell)
    if not result.success:
        print("⚠️ cell failed — continuing to the next cell")


In [29]:
import importlib
import inspect
import typing

import langchain_community.document_loaders
import pandas as pd
from langchain_core.stores import BaseStore
from langchain.storage import InMemoryStore
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma, VectorStore

from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_core.documents import Document
from langchain_core.language_models import BaseChatModel
from langchain_core.language_models.base import LanguageModelInput
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import (
    CommaSeparatedListOutputParser,
    JsonOutputParser,
)
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import langchain.chains.combine_documents as pkg


# COMANDOS PARA TYPE TETRIS

Objetivo: dado um `target` qualquer, descobrir sua estrutura (tipos, assinaturas, campos obrigatórios) respeitando o que cada comando de introspecção espera receber.

Cada comando de `inspect`/`typing` só aceita uma categoria de objeto. Regra prática:

| Categoria | Como chegar até ela (a partir de uma instância) | Comandos válidos |
|---|---|---|
| MODULE | `import module` | `dir(mod)`, `mod.__all__`, `typing.get_type_hints(mod)` |
| CLASS | `cls = type(target)` | `inspect.signature(cls)`, `typing.get_type_hints(cls.__init__)`, `inspect.getmro(cls)`, `cls.model_fields` (só se pydantic v2) |
| FUNCTION / UNBOUND METHOD | `fn = cls.__init__` | `inspect.signature(fn)`, `typing.get_type_hints(fn)` |
| BOUND METHOD | `bm = target.load` | `inspect.signature(bm)`, `bm.__func__`, `hasattr(bm, '__func__')` |
| INSTANCE | `target` em si | `dir(target)`, `vars(target)`, acesso direto a atributos |

In [4]:
%%catch
pdf_loader  = PyPDFLoader("../data/LangChain.pdf")
pdf_file = pdf_loader.load()

# QUICK PROBE (opcional, antes de decidir o `target` final)
Antes de decidir o `target`, cheque se tem required atributes.

In [30]:
%%catch
# The required/optional distinction is encoded implicitly in whether = default appears after the parameter:
# Use ChatOpenAI ou ChatOpenAI().invoke - sem () — não executa
probe = pkg


print(probe)
print("\n", inspect.signature(probe).parameters)
print("\n", inspect.signature(probe)._return_annotation)
# print("\n", inspect.signature(probe).__validate_parameters__)

<module 'langchain.chains.combine_documents' from '/Users/marcelohanones/Developer/personal/IBM-RAG-and-Agentic-AI/.venv/lib/python3.12/site-packages/langchain/chains/combine_documents/__init__.py'>


TypeError: <module 'langchain.chains.combine_documents' from '/Users/marcelohanones/Developer/personal/IBM-RAG-and-Agentic-AI/.venv/lib/python3.12/site-packages/langchain/chains/combine_documents/__init__.py'> is not a callable object

⚠️ cell failed — continuing to the next cell


# TARGET

In [32]:
# target = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
# target = loader.load()
# target = web_data

target = pkg

# target = xxxxxx.WebBaseLoader() -> instancia
# target = xxxxxx.WebBaseLoader -> classe

In [33]:
%%catch
print(type(target))  # -> a CLASS. target em si é a INSTANCE dessa classe.
print("\n", target)

<class 'module'>

 <module 'langchain.chains.combine_documents' from '/Users/marcelohanones/Developer/personal/IBM-RAG-and-Agentic-AI/.venv/lib/python3.12/site-packages/langchain/chains/combine_documents/__init__.py'>


# 1. MODULE
Ponto de partida: o módulo onde a classe mora.

In [34]:
%%catch
leaf_mod = importlib.import_module(type(target).__module__)  # módulo-FOLHA: onde a classe foi definida
print(leaf_mod.__name__)

builtins


In [35]:
%%catch
# dir(mod) mistura tudo — classes, funções, variáveis, imports. Pra pegar só as classes irmas:
[name for name, obj in vars(leaf_mod).items() if inspect.isclass(obj)]

['__loader__',
 'bool',
 'memoryview',
 'bytearray',
 'bytes',
 'classmethod',
 'complex',
 'dict',
 'enumerate',
 'filter',
 'float',
 'frozenset',
 'property',
 'int',
 'list',
 'map',
 'object',
 'range',
 'reversed',
 'set',
 'slice',
 'staticmethod',
 'str',
 'super',
 'tuple',
 'type',
 'zip',
 'BaseException',
 'BaseExceptionGroup',
 'Exception',
 'GeneratorExit',
 'KeyboardInterrupt',
 'SystemExit',
 'ArithmeticError',
 'AssertionError',
 'AttributeError',
 'BufferError',
 'EOFError',
 'ImportError',
 'LookupError',
 'MemoryError',
 'NameError',
 'OSError',
 'ReferenceError',
 'RuntimeError',
 'StopAsyncIteration',
 'StopIteration',
 'SyntaxError',
 'SystemError',
 'TypeError',
 'ValueError',
 'Warning',
 'FloatingPointError',
 'OverflowError',
 'ZeroDivisionError',
 'BytesWarning',
 'DeprecationWarning',
 'EncodingWarning',
 'FutureWarning',
 'ImportWarning',
 'PendingDeprecationWarning',
 'ResourceWarning',
 'RuntimeWarning',
 'SyntaxWarning',
 'UnicodeWarning',
 'UserWarning

> Pacote pai — a família de classes-irmãs <br>
>> `type(target).__module__` sempre aponta pro módulo-folha onde a classe foi *definida* — normalmente só ela mais suas dependências importadas, como visto acima. 
>>Pra ver todas as classes relacionadas (ex: os 197 `*Loader` de `langchain_community.document_loaders`), sobe um nível via `__package__`.

In [36]:
%%catch
pkg = importlib.import_module(leaf_mod.__package__)   # agnóstico: sobe a partir de leaf_mod, não hardcoded
print(pkg.__name__, "-", len(pkg.__all__), "nomes exportados")

sorted(pkg.__all__)  # amostra — a lista completa tem 197 entradas

ValueError: Empty module name

⚠️ cell failed — continuing to the next cell


In [37]:
%%catch
# pkg usa lazy-loading via __getattr__: cada getattr(pkg, nome) importa o submódulo daquele
# loader na hora. Fazer isso pra TODAS as 197 entradas é lento e pode falhar em quem exige
# dependência extra não instalada (ex: airbyte-cdk). Verificamos só a nossa classe, como prova:
inspect.isclass(getattr(pkg, type(target).__name__))   # True — confirma que a entrada em __all__ é mesmo uma classe
# (cls só é definido na seção 2 — aqui usamos type(target) direto pelo mesmo motivo)

AttributeError: module 'langchain.chains.combine_documents' has no attribute 'module'

⚠️ cell failed — continuing to the next cell


In [38]:
%%catch
print(f'''O pacote: {pkg.__name__} \n
O módulo: {leaf_mod.__name__} \n
A classe: {type(target).__name__}''')

O pacote: langchain.chains.combine_documents 

O módulo: builtins 

A classe: module


# 2. CLASS
> target é a instancia. (com ())
>> cls = type(target) → agora cls aponta pra classe, não pro objeto. 
>>> É aqui que moram assinatura do construtor, MRO e (se pydantic v2) `model_fields`.

In [39]:
%%catch
cls = type(target)
cls

module

In [40]:
%%catch
# A Assinatura do Construtor não retorna métodos — retorna os parâmetros que você precisa passar pra construir a instância, não necessariamente "quais atributos a instância vai ter depois" — nesse caso específico (Document) as duas coisas coincidem, mas isso é uma propriedade do pydantic, não uma regra geral do Python.

# Se pydantic , este parâmetro fica guardado como um atributo de instância — uma string pura. Você acessa sem () pra pegar o texto. 

# Nunca assuma que a assinatura do construtor (seção 2/CLASS) é a lista final de atributos da instância — isso só é garantido quando você já confirmou que a classe é pydantic e não usa alias/validator/private attr. Fora isso, a seção 5 (INSTANCE, com dir()/vars() na instância já construída) é a única fonte confiável do que realmente existe.

sig = inspect.signature(cls)     
pd.DataFrame([
    {
        "name": name,
        "kind": p.kind.name,
        "default": p.default,
        # *args/**kwargs sempre reportam default=empty (não existe "default" pra eles),
        # mas isso não os torna obrigatórios — podem ser passados vazios.
        "required": p.default is inspect.Parameter.empty and p.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD),
        "annotation": p.annotation,
    }
    for name, p in sig.parameters.items()
])

,name,kind,default,required,annotation
0,name,POSITIONAL_OR_KEYWORD,<class 'inspect._empty'>,True,<class 'inspect._empty'>
1,doc,POSITIONAL_OR_KEYWORD,None,False,<class 'inspect._empty'>


### O que tem dentro do `**kwargs`?
`**kwargs` na assinatura acima é uma caixa-preta — `inspect.signature(cls)` não revela o que está lá dentro. Geralmente a subclasse só repassa (`super().__init__(**kwargs)`) pro `__init__` da classe-mãe na MRO (seção 2, célula `c6b41693`). Pra ver os parâmetros reais, precisa inspecionar o `__init__` de cada ancestral que define o seu próprio (não herdado).

In [41]:
%%catch
def full_init_params(cls):
    seen = {}
    for klass in inspect.getmro(cls):
        if "__init__" not in vars(klass):   # só __init__ definido NESSA classe, não herdado
            continue
        for name, p in inspect.signature(klass.__init__).parameters.items():
            if name in ("self", "args", "kwargs"):
                continue
            seen.setdefault(name, (klass.__name__, p))   # primeira ocorrência = mais específica na MRO
    return seen

for name, (owner, p) in full_init_params(cls).items():
    flag = " <-- accepts **kwargs-style unpacking" if p.kind == inspect.Parameter.VAR_KEYWORD else ""
    print(f"{name:20} from {owner:20} kind={p.kind.name:15} default={p.default!r}{flag}")

# This function bypasses normal Python attribute lookup — it manually walks every class in inspect.getmro(cls) and grabs the __init__ defined directly on each one (via vars(klass)), not just the first one found. That's how it reaches all the way down to pulled out its real parameters.

In [42]:
%%catch
hints = typing.get_type_hints(cls.__init__)   # tipos declarados no __init__
pd.DataFrame(hints.items(), columns=["name", "type_hint"])

# Para classes pydantic, pode ocorrer get_type_hints nao listar todos os campos.Nestes casos, model_fields é a fonte confiável dos campos — get_type_hints(cls.__init__) só é confiável quando o __init__ não foi customizado à mão (o que é comum em wrappers de conveniência como WebBaseLoader.load()).

,name,type_hint


In [43]:
%%catch
inspect.getmro(cls)                            # cadeia de herança REAL da classe (não da metaclass)

(module, object)

In [44]:
%%catch
# model_fields só existe quando a CLASS é um pydantic v2 BaseModel de verdade
# (metaclass ModelMetaclass). 
is_pydantic_v2 = type(cls).__name__ == "ModelMetaclass"
print(f"{cls.__name__}: metaclass={type(cls).__name__}, pydantic_v2={is_pydantic_v2} \n")

if is_pydantic_v2:
    for name, info in cls.model_fields.items():
        args = typing.get_args(info.annotation)
        nullable = type(None) in args
        print(f"{name:19} required={info.is_required()!s:6} nullable={nullable!s:6} default={info.default!r}")
else:
    print(f"{cls.__name__} não é pydantic v2 — sem model_fields. Use a assinatura do construtor (célula acima).")

module: metaclass=type, pydantic_v2=False 

module não é pydantic v2 — sem model_fields. Use a assinatura do construtor (célula acima).


# 3. FUNCTION / UNBOUND METHOD
`cls.__init__` acessado pela CLASS (não pela instância) é uma função pura — ainda não tem `self` amarrado.

In [45]:
%%catch
fn = cls.__init__
print(inspect.signature(fn))          # inclui `self` — ainda não amarrado

(self, /, *args, **kwargs)


In [46]:
%%catch
typing.get_type_hints(fn)             # mesmos tipos que via cls.__init__ na seção anterior

{}

In [47]:
%%catch
hasattr(fn, "__func__")               # False: função pura acessada pela classe não é bound method

False

# 4. BOUND METHOD
Um método acessado pela INSTANCE já vem com `self` amarrado — por isso `inspect.signature` nunca mostra `self` aqui, e `__func__` aponta de volta pra função pura da classe.

In [48]:
%%catch
# Acha, entre tudo que target tem, um método que pertence a ELE (não à classe) — e guarda esse método pronto-pra-usar em bm.

methods = inspect.getmembers(target, predicate=inspect.ismethod)
# filtra classmethods (self=classe) — só queremos bound methods de verdade (self=target)
instance_methods = [(n, m) for n, m in methods if not n.startswith("_") and m.__self__ is target]
method_name, bm = instance_methods[0]   # primeiro bound method de instância, não hardcoded
print(method_name)
bm

IndexError: list index out of range

⚠️ cell failed — continuing to the next cell


In [49]:
%%catch
# olha esse bm de três ângulos: o que falta passar pra chamá-lo, se ele é mesmo um "método amarrado", e se por baixo é a mesma função que a classe define.

print(inspect.signature(bm))   # sem `self` na lista de params
print(hasattr(bm, "__func__"))  # True — só serve para bound method (ou functools.partial) 
print(bm.__func__ is getattr(cls, method_name))  # True: é a mesma função, só que "amarrada" à instância

NameError: name 'bm' is not defined

⚠️ cell failed — continuing to the next cell


> `__self__` é o teste real de bound method
>>`callable()` dá falso positivo (classe crua também é callable). 
>>>`hasattr(x, >>>"__func__")` dá falso negativo em métodos built-in (`dict.keys`). 
>>>>`hasattr(x, "__self__")` é o único que acerta os três casos.

In [50]:
%%catch
def find_builtin_bound_method(obj):
    # 1) direto na instância (raro, mas possível em objetos C-extension)
    for name, val in inspect.getmembers(obj, predicate=inspect.isbuiltin):
        return name, val
    # 2) um nível abaixo: dentro de containers builtin que a instância guarda
    for name in dir(obj):
        if name.startswith("_"):
            continue
        try:
            val = getattr(obj, name)
        except Exception:
            continue
        if isinstance(val, (dict, list, set, tuple, frozenset)):
            for mname, mval in inspect.getmembers(val, predicate=inspect.isbuiltin):
                return f"{name}.{mname}", mval
    return None, None

md_label, md_keys = find_builtin_bound_method(target)   # agnóstico: acha um builtin bound method onde quer que ele esteja (ou None)

# __self__ é o teste real de bound method — callable() dá falso positivo (cls), __func__ dá falso negativo (built-in)
casos = {"bm": bm, f"{md_label} (builtin)": md_keys, "cls (classe crua)": cls}
for label, obj in casos.items():
    print(f"{label:20} callable={callable(obj)!s:6} __self__={hasattr(obj, '__self__')!s:6} __func__={hasattr(obj, '__func__')}")

print()
for label, obj in casos.items():
    veredito = "É bound method" if hasattr(obj, "__self__") else "NÃO é bound method"
    print(f"{label:20} -> {veredito}")

NameError: name 'bm' is not defined

⚠️ cell failed — continuing to the next cell


> Distincao entre Instance e Bound Method

In [51]:
%%catch
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
web_data = loader.load()

In [52]:
%%catch
web_data = loader.load() # web_data is an instance, a class instance of what load() returns, what happen to be a list. So web_data is a class instance of a list.

web_data = loader.load # web_data is a bound method do mesmo valor que loader.load.  Mas web_data e loader.load sao dois objetos distintos na memoria, embora ambos envolvem o mesmo self (loader) e a mesma funcao (WebBaseLoader.load)
print(web_data is loader.load)     # False — cada acesso a .load cria um wrapper novo
print(web_data == loader.load)     # True  — comparam __self__ e __func__, que são os mesmos

False
True


# 5. INSTANCE
`target` em si. Não é module, class, function nem method — é dado. `inspect.signature`, `typing.get_type_hints`, `len()`, `.model_fields`, `.keywords` NÃO se aplicam aqui (a menos que a classe implemente `__call__`, `__len__` etc). O que funciona é `dir()`, `vars()` e acesso direto a atributos/métodos concretos.

In [53]:
%%catch
# mostra o objeto cru, sem filtro nenhum — a "foto" bruta da instância antes de qualquer investigação
target

<module 'langchain.chains.combine_documents' from '/Users/marcelohanones/Developer/personal/IBM-RAG-and-Agentic-AI/.venv/lib/python3.12/site-packages/langchain/chains/combine_documents/__init__.py'>

In [54]:
%%catch
#DADOS

pd.set_option("display.max_colwidth", 80)

def describe_dado(value):
    r = repr(value)
    if len(r) <= 60:
        return r
    if isinstance(value, str):
        return f"str ({len(value)} chars)"
    if isinstance(value, dict):
        return f"dict ({len(value)} keys)"
    if isinstance(value, (list, tuple, set)):
        return f"{type(value).__name__} ({len(value)} items)"
    return f"{type(value).__name__} (repr longo: {len(r)} chars)"

def align_left(df):   # header + células alinhados à esquerda, pra ler tudo retinho
    return df.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": [("text-align", "left")]}]
    )

dados_rows, metodos_rows = [], []
for name in dir(target):
    if name.startswith("_"):
        continue
    value = getattr(target, name)
    if callable(value):
        self_repr = type(value.__self__).__name__ if hasattr(value, "__self__") else "?"
        metodos_rows.append({"name": name, "method": f"{self_repr}.{name}"})
    else:
        printed = str(value)
        dados_rows.append({
            "name": name,
            "resumo": describe_dado(value)[:15],
            "print[:20]": printed[:20],
            "len": len(printed),
        })

# colunas explícitas: evita KeyError no sort_values quando dados_rows vem vazio
# (ex: instância cujos atributos públicos são só métodos, como CharacterTextSplitter)
dados_df = pd.DataFrame(dados_rows, columns=["name", "resumo", "print[:20]", "len"])
if not dados_df.empty:
    dados_df = dados_df.sort_values("name").reset_index(drop=True)

align_left(dados_df)

,name,resumo,print[:20],len
0,base,module (repr lo,<module 'langchain.c,199
1,map_reduce,module (repr lo,<module 'langchain.c,211
2,map_rerank,module (repr lo,<module 'langchain.c,211
3,reduce,module (repr lo,<module 'langchain.c,203
4,refine,module (repr lo,<module 'langchain.c,203
5,stuff,module (repr lo,<module 'langchain.c,201


In [55]:
%%catch
metodos_df = pd.DataFrame(metodos_rows).sort_values("name").reset_index(drop=True)
align_left(metodos_df)

,name,method
0,acollapse_docs,?.acollapse_docs
1,collapse_docs,?.collapse_docs
2,create_stuff_documents_chain,?.create_stuff_documents_chain
3,split_list_of_docs,?.split_list_of_docs
